In [0]:
dbutils.widgets.text("base_path", "/Volumes/swiggy/pipeline/raw_data")

In [0]:
BASE_PATH = dbutils.widgets.get("base_path")

LANDING_DELIVERY = BASE_PATH + "/landing/delivery"
CHECKPOINT_DELIVERY = BASE_PATH + "/checkpoints/delivery"
BRONZE_DELIVERY_TABLE       = "swiggy.bronze.delivery_raw"

print(f"BASE_PATH       : {BASE_PATH}")
print(f"LANDING_DELIVERY  : {LANDING_DELIVERY}")
print(f"CHECKPOINT_DELIVERY: {CHECKPOINT_DELIVERY}")
print(f"BRONZE_DELIVERY   : {BRONZE_DELIVERY}")

In [0]:
from pyspark.sql.types import StructField , StructType , StringType , DoubleType , IntegerType , BooleanType

delivery_schema = StructType([
  StructField("delivery_id" , StringType() , False),
  StructField("order_id" , StringType() , False),
  StructField("agent_id" , StringType() , False),
  StructField("restaurant_id" , StringType() , False),
  StructField("pickup_lat", DoubleType() , True),
  StructField("pickup_lon", DoubleType() , True),
  StructField("drop_lat", DoubleType() , True),
  StructField("drop_lon", DoubleType() , True),
  StructField("promised_time_min", IntegerType() , True),
  StructField("actual_time_min" , IntegerType() , True),
  StructField("distance_km" , DoubleType() , False),
  StructField("is_late" , BooleanType() , False),
  StructField("status" , StringType() , False),
  StructField("city" , StringType() , False),
  StructField("created_at" , StringType() , False)
])


In [0]:
raw_df = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.schemaLocation", CHECKPOINT_DELIVERY + "/schema") \
  .schema(delivery_schema) \
  .load(LANDING_DELIVERY)

from pyspark.sql.functions import col, current_date, current_timestamp, to_date, to_timestamp

bronze_df = (raw_df.withColumn("bronze_ingested_at", current_timestamp()) \
  .withColumn("bronze_ingest_date", current_date()) \
  .withColumn("bronze_source_file" , col("_metadata.file_path")))


print("schema_ready")

In [0]:
( bronze_df.writeStream
           .format("delta")
           .option("checkpointLocation", CHECKPOINT_DELIVERY)
           .outputMode("append")
           .trigger(availableNow=True)
           .toTable(BRONZE_DELIVERY_TABLE)
           .awaitTermination() )


In [0]:
bronze_df_check = spark.read.format("delta").load(BRONZE_DELIVERY)
print(f"Total rows: {bronze_df_check.count()}")
bronze_df_check.show(5, truncate=False)
bronze_df_check.printSchema()